# Trotter product formulas

Approximate evolution under a sum of Hamiltonian terms by specifying their
application order and coefficients. For the second-order sequence $X,ZZ,X$,
each pair of index and coefficient contributes a native rotation angle of
$2 c_k t$. Each term acts on the same ordered through register.

Adapted from the [Qualtran v0.7.0 notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/trotter/trotterized_unitary.ipynb), Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import Markdown, SVG, display
import torch

from flagquantum.algorithms.primitives.ising_evolution import IsingXUnitary, IsingZZUnitary
from flagquantum.algorithms.primitives.trotter_product import TrotterProduct
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.drawer.signature_graph import export_signature_graph_dot
from flagquantum.qec.resources.estimation import estimate_ftqc_resources
from flagquantum.simulation.unitary import get_unitary

## Three-site second-order example

Use transverse coupling 0.1, ZZ coupling 2 and a timestep of 0.01.
The two outer X layers each use half the transverse coupling.
The Ising terms receive their angles from the product schedule.

In [ ]:
sites, gamma, coupling, timestep = 3, 0.1, 2, 0.01
wires = tuple(range(sites))
x, zz = IsingXUnitary(sites), IsingZZUnitary(sites)
product = TrotterProduct((x, zz), (0, 1, 0), (gamma / 2, coupling, gamma / 2), timestep)
display(SVG(data=render_call_graph_svg(
    export_signature_graph_dot(product.signature, "TrotterizedUnitary", pretty=True)
).decode()))

## Immediate calls and leaf counts

Both X layers use the same numerical angle, so their contributions
combine into a single edge of multiplicity two in the call graph.

In [ ]:
model = product.resource_model()
graph = modeled_call_graph(model, max_depth=1)
display(SVG(data=render_call_graph_svg(
    export_modeled_call_graph_dot(model, max_depth=1)
).decode()))
display(Markdown(format_call_graph_markdown(
    graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name
)))
assert {node.name: count for node, count in graph.leaf_counts.items()} == {
    "IsingXUnitary": 2, "IsingZZUnitary": 1,
}

## Execute the product

Append the product to a circuit and compare its full complex matrix to
the matrix product of the individually bound terms. The two reports
describe emitted instructions and model resources respectively.

In [ ]:
op = product.make_on(system=wires)
circuit = Circuit(sites).append(op)
ux = get_unitary(Circuit(sites).append(x.make_on(system=wires, angle=gamma * timestep)),
                 dtype=torch.complex128)
uzz = get_unitary(Circuit(sites).append(zz.make_on(system=wires, angle=2 * coupling * timestep)),
                  dtype=torch.complex128)
expected = ux @ uzz @ ux
torch.testing.assert_close(get_unitary(circuit, dtype=torch.complex128), expected, atol=1e-12, rtol=0)
inverse = Circuit(sites).append(product.make_on(system=wires, adjoint=True))
torch.testing.assert_close(get_unitary(inverse, dtype=torch.complex128), expected.mH, atol=1e-12, rtol=0)
report = estimate_ftqc_resources(op, emitted_ir=circuit.to_ir())
assert report.emitted is not None
assert report.emitted.unconditional == {"cx": 6, "rx": 6, "rz": 3}
assert report.logical.gates.rotation == 9
assert report.logical.peak_logical_qubits == 3
print("Emitted instructions:", report.emitted.unconditional)
print("Modeled logical qubits:", report.logical.peak_logical_qubits)